# JeevanSetu: train the wound-photo model (MobileNetV2 on the AZH dataset)

This notebook follows Table V of our paper. It downloads the public **AZH wound dataset**, trains **MobileNetV2** with two-stage transfer learning, reports the test results (accuracy, macro-F1, coverage and selective accuracy with abstention), and exports the files the JeevanSetu app needs.

**Run it on Kaggle (free GPU):**
1. kaggle.com → **Create → New Notebook**, then **File → Import Notebook** and upload this file.
2. In the right panel: **Session options → Accelerator → GPU T4 x2** (or P100), and turn **Internet on**. Kaggle asks for phone verification once before it allows internet.
3. **Run All**. Training takes about 5–10 minutes on a GPU.
4. Download **`jeevansetu_model.zip`** from the **Output** panel (`/kaggle/working`).

**Google Colab also works:** Runtime → Change runtime type → T4 GPU, then Runtime → Run all, and download the zip from the Files panel.

**Using the result in the app:** unzip it and copy `wound_model.onnx` and `wound_model.json` into `JeevanSetu_Project/server/models/`, then restart the app. `mobilenetv2_azh.pt` is for the Python backend (`backend/ml/weights/`).

In [ ]:
!pip -q install timm onnx onnxruntime scikit-learn

In [ ]:
import io, json, os, random, shutil, time, urllib.request, zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import timm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
CLASSES = ["diabetic", "pressure", "surgical", "venous"]
LETTER_TO_CLASS = {"D": "diabetic", "P": "pressure", "S": "surgical", "V": "venous"}  # BG and N are not used
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
DATA = Path("/tmp/azh_data")  # kept outside /kaggle/working so the Output panel only shows results
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RUN_COMPARISON = DEVICE == "cuda"  # also trains EfficientNet-B0 and ResNet-50, as in Table VI of the paper

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
OUT.mkdir(parents=True, exist_ok=True)
print("device:", DEVICE, "| torch", torch.__version__, "| timm", timm.__version__)

## 1. Download the AZH dataset and arrange the folders

In [ ]:
REPO_ZIP = "https://codeload.github.com/uwm-bigdata/Multi-modal-wound-classification-using-images-and-locations/zip/refs/heads/main"

def prepare_dataset(target=DATA):
    if (target / "train").exists():
        return target
    raw = Path("/tmp/azh_raw"); raw.mkdir(exist_ok=True)
    with urllib.request.urlopen(REPO_ZIP) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(raw)
    dataset_dir = next(raw.glob("*/dataset"))
    for split, name in [("train", "Train.zip"), ("test", "Test.zip")]:
        tmp = raw / split
        zipfile.ZipFile(dataset_dir / name).extractall(tmp)
        source = next(p for p in tmp.iterdir() if p.is_dir() and not p.name.startswith("__"))
        for letter, cls in LETTER_TO_CLASS.items():
            dest = target / split / cls
            dest.mkdir(parents=True, exist_ok=True)
            for f in (source / letter).iterdir():
                if f.suffix.lower() in {".jpg", ".jpeg", ".png"} and not f.name.startswith("."):
                    shutil.copy(f, dest / f.name)
    return target

prepare_dataset()
for split in ["train", "test"]:
    counts = {c: len(list((DATA / split / c).iterdir())) for c in CLASSES}
    print(split, counts, "total", sum(counts.values()))

## 2. Train / validation split and data loaders (Table V)

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15), transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

base = datasets.ImageFolder(DATA / "train")
assert base.classes == CLASSES, base.classes

def stratified_split(targets, fraction=0.15, seed=SEED):
    rng = np.random.default_rng(seed)
    train_idx, val_idx = [], []
    for c in sorted(set(targets)):
        idx = [i for i, t in enumerate(targets) if t == c]
        rng.shuffle(idx)
        cut = round(len(idx) * fraction)
        val_idx += idx[:cut]; train_idx += idx[cut:]
    return train_idx, val_idx

train_idx, val_idx = stratified_split(base.targets)
train_set = Subset(datasets.ImageFolder(DATA / "train", train_tf), train_idx)
val_set = Subset(datasets.ImageFolder(DATA / "train", eval_tf), val_idx)
test_set = datasets.ImageFolder(DATA / "test", eval_tf)
print(f"train {len(train_set)}, validation {len(val_set)}, test {len(test_set)}")

workers = 2 if DEVICE == "cuda" else 0
train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=workers)
val_loader = DataLoader(val_set, batch_size=64, num_workers=workers)
test_loader = DataLoader(test_set, batch_size=64, num_workers=workers)

counts = np.bincount([base.targets[i] for i in train_idx], minlength=len(CLASSES))
class_weights = torch.tensor(counts.sum() / (len(CLASSES) * counts), dtype=torch.float32)  # Eq. (3)
print("class weights", class_weights.numpy().round(3))

## 3. Two-stage transfer learning

In [ ]:
from sklearn.metrics import f1_score

@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, labels = [], []
    for x, y in loader:
        probs.append(torch.softmax(model(x.to(DEVICE)), 1).cpu()); labels.append(y)
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()

def run_epochs(model, optimizer, criterion, epochs, patience, state):
    stale = 0
    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            optimizer.zero_grad()
            criterion(model(x.to(DEVICE)), y.to(DEVICE)).backward()
            optimizer.step()
        p, y = predict(model, val_loader)
        f1 = f1_score(y, p.argmax(1), average="macro")
        print(f"  epoch {epoch + 1:2d}: validation macro-F1 {f1:.3f}")
        if f1 > state["best_f1"]:
            state.update(best_f1=f1, best={k: v.detach().clone() for k, v in model.state_dict().items()}); stale = 0
        else:
            stale += 1
            if patience and stale >= patience:
                print("  early stop"); return

def train(arch):
    set_seed()
    model = timm.create_model(arch, pretrained=True, num_classes=len(CLASSES), drop_rate=0.2).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    head = {id(p) for p in model.get_classifier().parameters()}
    state = {"best_f1": -1.0, "best": None}
    started = time.time()
    print(f"{arch}: stage 1, backbone frozen")
    for p in model.parameters():
        p.requires_grad = id(p) in head
    run_epochs(model, torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-3), criterion, 5, 0, state)
    print(f"{arch}: stage 2, last 30% of parameter tensors unfrozen")
    params = list(model.parameters())
    for p in params[int(len(params) * 0.7):]:
        p.requires_grad = True
    model.load_state_dict(state["best"])
    run_epochs(model, torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-4), criterion, 12, 4, state)
    model.load_state_dict(state["best"])
    print(f"{arch}: trained in {(time.time() - started) / 60:.1f} min, best validation macro-F1 {state['best_f1']:.3f}")
    return model, state["best_f1"]

model, val_f1 = train("mobilenetv2_100")

## 4. Test results

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support

val_probs, _ = predict(model, val_loader)
tau = float(np.percentile(val_probs.max(1), 15))  # abstention threshold fixed on validation data
test_probs, test_y = predict(model, test_loader)
pred = test_probs.argmax(1)
answered = test_probs.max(1) >= tau

def cpu_latency_ms(m, runs=50):
    m = m.to("cpu").eval(); torch.set_num_threads(1)
    x = torch.randn(1, 3, 224, 224); times = []
    with torch.no_grad():
        for _ in range(5): m(x)
        for _ in range(runs):
            t = time.perf_counter(); m(x); times.append((time.perf_counter() - t) * 1000)
    m.to(DEVICE)
    return float(np.median(times))

p, r, f, s = precision_recall_fscore_support(test_y, pred, labels=range(len(CLASSES)), zero_division=0)
metrics = {
    "test_accuracy": round(float(accuracy_score(test_y, pred)), 4),
    "test_macro_f1": round(float(f1_score(test_y, pred, average="macro")), 4),
    "validation_macro_f1": round(float(val_f1), 4),
    "tau": round(tau, 4),
    "coverage": round(float(answered.mean()), 4),
    "selective_accuracy": round(float((pred[answered] == test_y[answered]).mean()), 4),
    "cpu_ms_per_image": round(cpu_latency_ms(model), 1),
    "parameters_millions": round(sum(q.numel() for q in model.parameters()) / 1e6, 2),
    "per_class": {c: {"precision": round(float(p[i]), 3), "recall": round(float(r[i]), 3), "f1": round(float(f[i]), 3), "support": int(s[i])} for i, c in enumerate(CLASSES)},
    "images": {"train": len(train_set), "validation": len(val_set), "test": len(test_set)},
}
print(json.dumps({k: v for k, v in metrics.items() if k != "per_class"}, indent=1))
print(classification_report(test_y, pred, target_names=CLASSES, digits=3))
print("confusion matrix (rows = true, columns = predicted)\n", confusion_matrix(test_y, pred))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
cm = confusion_matrix(test_y, pred)
axes[0].imshow(cm, cmap="Greys")
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        axes[0].text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
axes[0].set_xticks(range(4), [c[:5].title() + "." for c in CLASSES]); axes[0].set_yticks(range(4), [c[:5].title() + "." for c in CLASSES])
axes[0].set_xlabel("Predicted class"); axes[0].set_ylabel("True class"); axes[0].set_title("(a) Confusion matrix (test set)")

conf = test_probs.max(1); coverages, accs = [], []
for q in np.linspace(0, 50, 11):
    t = np.percentile(conf, q); keep = conf >= t
    coverages.append(keep.mean() * 100); accs.append((pred[keep] == test_y[keep]).mean() * 100)
axes[1].plot(coverages, accs, "o-", color="black")
axes[1].plot([metrics["coverage"] * 100], [metrics["selective_accuracy"] * 100], "s", color="#0F766E", markersize=10, label=f"operating point, tau = {tau:.2f}")
axes[1].invert_xaxis(); axes[1].set_xlabel("Coverage (%)"); axes[1].set_ylabel("Selective accuracy (%)")
axes[1].set_title("(b) Coverage vs. selective accuracy"); axes[1].grid(alpha=0.3); axes[1].legend()
plt.tight_layout(); plt.savefig(OUT / "results.png", dpi=200); plt.show()

## 5. Optional: compare with EfficientNet-B0 and ResNet-50 (Table VI of the paper)

In [ ]:
comparison = [{"backbone": "MobileNetV2", "params_m": metrics["parameters_millions"], "val_macro_f1": metrics["validation_macro_f1"],
               "test_accuracy": metrics["test_accuracy"], "test_macro_f1": metrics["test_macro_f1"], "cpu_ms": metrics["cpu_ms_per_image"]}]
if RUN_COMPARISON:
    for arch, label in [("efficientnet_b0", "EfficientNet-B0"), ("resnet50", "ResNet-50")]:
        other, other_val = train(arch)
        pr, yy = predict(other, test_loader)
        comparison.append({"backbone": label, "params_m": round(sum(q.numel() for q in other.parameters()) / 1e6, 2), "val_macro_f1": round(other_val, 4),
                           "test_accuracy": round(float((pr.argmax(1) == yy).mean()), 4), "test_macro_f1": round(float(f1_score(yy, pr.argmax(1), average="macro")), 4),
                           "cpu_ms": round(cpu_latency_ms(other), 1)})
        del other
metrics["comparison"] = comparison
for row in comparison:
    print(row)

## 6. Export for the JeevanSetu app

In [ ]:
class ExportWrapper(nn.Module):
    # Returns class scores and the last feature map, so the app can draw Grad-CAM without gradients.
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, image):
        features = self.m.forward_features(image)
        return self.m.forward_head(features), features

export_model = ExportWrapper(model.to("cpu").eval()).eval()
dummy = torch.randn(1, 3, 224, 224)
onnx_path = OUT / "wound_model.onnx"
kwargs = dict(input_names=["image"], output_names=["logits", "features"], opset_version=17)
try:
    torch.onnx.export(export_model, dummy, onnx_path, dynamo=False, **kwargs)
except TypeError:
    torch.onnx.export(export_model, dummy, onnx_path, **kwargs)

import onnxruntime as ort
session = ort.InferenceSession(str(onnx_path))
onnx_logits, onnx_features = session.run(None, {"image": dummy.numpy()})
with torch.no_grad():
    torch_logits, torch_features = export_model(dummy)
print("max difference torch vs onnx:", float(np.abs(onnx_logits - torch_logits.numpy()).max()), "| feature map", onnx_features.shape)

classifier = model.get_classifier()
meta = {
    "name": "mobilenetv2_azh", "version": "1", "classes": CLASSES, "tau": round(tau, 4), "input_size": 224,
    "mean": MEAN, "std": STD,
    "classifier_weight": classifier.weight.detach().cpu().numpy().round(6).tolist(),
    "classifier_bias": classifier.bias.detach().cpu().numpy().round(6).tolist(),
    "metrics": metrics,
    "trained_at": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
    "dataset": "AZH wound dataset (UWM BigData Lab), official train/test split, 15% of train held out for validation",
}
(OUT / "wound_model.json").write_text(json.dumps(meta))
torch.save({"state_dict": model.state_dict(), "classes": CLASSES, "tau": tau, "version": "1"}, OUT / "mobilenetv2_azh.pt")

with open(OUT / "metrics.txt", "w") as fh:
    fh.write(json.dumps(metrics, indent=2))
with zipfile.ZipFile(OUT / "jeevansetu_model.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["wound_model.onnx", "wound_model.json", "mobilenetv2_azh.pt", "results.png", "metrics.txt"]:
        z.write(OUT / name, name)
print("Done. Download", OUT / "jeevansetu_model.zip")
print(f"Test accuracy {metrics['test_accuracy']:.1%}, macro-F1 {metrics['test_macro_f1']:.3f}, "
      f"coverage {metrics['coverage']:.1%} at tau {tau:.2f}, selective accuracy {metrics['selective_accuracy']:.1%}")